# Functions, Parameter Mechanics & Call Stacks

## Lesson 1.4.2: Scopes, Name Resolution (LEGB) & Variable Binding

### Core Mechanics & Theory

When Python encounters a variable name (identifier), it resolves it using the **LEGB rule**. Understanding this lookup order and how Python compiles scope bindings is essential before building closures, decorators, and state machines.

### LEGB Scope Hierarchy Diagram

```
+--------------------------------------------------------+
| Built-in Scope (B): print, len, range, ValueError...   |
|   +--------------------------------------------------+ |
|   | Global / Module Scope (G): Top-level variables   | |
|   |   +--------------------------------------------+ | |
|   |   | Enclosing / Nonlocal Scope (E): Outer func | | |
|   |   |   +--------------------------------------+ | | |
|   |   |   | Local Scope (L): Current func body   | | | |
|   |   |   +--------------------------------------+ | | |
|   |   +--------------------------------------------+ | |
|   +--------------------------------------------------+ |
+--------------------------------------------------------+
```

## 1. The LEGB Resolution Order

When Python looks up a variable name, it searches through scopes in this exact order:

- **L (Local)**: Names assigned or bound within the current active function frame
- **E (Enclosing)**: Names in the local scope of any enclosing/outer functions (closures)
- **G (Global)**: Names assigned at the top-level of the module file or declared `global`
- **B (Built-in)**: Preloaded names in Python's built-in namespace (`builtins` module)

**Important**: If a name is not found across all four scopes, Python raises `NameError`.

## 2. The Local Assignment Trap: UnboundLocalError

### Key Insight

**Python determines variable scope at compile time, not at runtime.**

If a function assigns to a variable anywhere in its body, Python marks that variable as **strictly local** for the entire function frame. This is decided before the function is executed.

### The Problem

Consider this broken code:

In [ ]:
# BROKEN CODE:
counter = 0

def increment():
    print(counter)  # Crashes with UnboundLocalError!
    counter = counter + 1

# Uncommenting this will raise: UnboundLocalError: local variable 'counter' referenced before assignment
# increment()

### Why This Crashes

1. **At compile time**, Python sees `counter = ...` inside `increment()` and marks `counter` as a **Local variable**

2. **At runtime**, Python executes `print(counter)` first
   - Because `counter` is marked as local, it skips the Global scope entirely
   - Python checks the local frame for `counter`
   - The local variable `counter` hasn't been assigned yet → **UnboundLocalError**

This is the classic "**Use Before Definition in Local Scope**" trap.

## 3. global vs. nonlocal

### global var_name

Explicitly instructs the compiler to bind assignments to the **Module/Global namespace** instead of creating a local variable.

**Use case**: Modify top-level module variables from within a function.

### nonlocal var_name

Introduced in **Python 3**. Instructs the compiler to bind assignments to the **nearest enclosing function scope** (skipping the local frame, but stopping before reaching the global module scope).

**Use case**: Modify variables from an outer function in nested function scenarios (closures).

In [ ]:
def outer():
    count = 0  # Enclosing scope variable
    
    def inner():
        nonlocal count  # Binds to outer()'s count
        count += 1
        return count
    
    # Call inner() multiple times
    print(inner())  # 1
    print(inner())  # 2
    print(inner())  # 3
    return count

result = outer()
print(f"Final count: {result}")  # Final count: 3